El taller tiene dos partes:

I) En la primera parte comenzaremos a trabajar con el concepto de Modulación, que sumado a lo visto en el Taller 3 completa los temas fundamentales/básicos de un sistema de comunicación. En particular en esta parte trabajaremos con Modulación AM. (Ejercicio 1)

II) En la parte dos exploraremos las implicancias del Teorema de Muestreo. (Ejercicio 2)


# **PARTE I**

## Introducción a la modulación. Modulación AM
Hemos visto que usando un filtro pasobajo configurado a cierta frecuencia de corte logramos que (aunque quizá no se escuche del todo bien) la señal tenga el ancho de banda que queremos. Por ejemplo, las radio AM tienen un ancho de banda de 10 kHz.
Pero si las radios AM transmitieran cada una el audio por la antena lo que podríamos recibir es todos los audios mezclados de todas las radios y no serviría de mucho. Para que podamos separar cada radio, cada estación de radio transmite en una cierta frecuencia ese audio, que llamaremos frecuencia portadora $f_c$. Es decir que lo que hace la radio es mover el espectro de frecuencia del audio que genera para que en lugar de estar entre 0 y 10 khz, esté entre $f_c$ y $f_c$ + 10 kHz .
A continuación veremos este proceso de mover el espectro a una frecuencia deseada que hacen las radios AM.





In [ ]:
import librosa
from IPython.display import Audio
# Ahora será necesario  leer e archivo wav que tenga disponible
# Para leer un archivo wav puede hacerlo de la siguiente forma:
# Deben sustituir Jaime_24.wav por el nombre del wav que ustedes hayan subido
data, fs = librosa.load('Jaime_24.wav')# le devuelve un numpy array con las muestras y la frecuencia de muestreo que utiliza esa canción.

#Escuche el archivo para ver como suena
Audio(data,rate=fs)

In [ ]:
# Repetimos aquí por comodidad el código para hacer el gráfico de la representación en frecuencia de una señal ya que lo utilizaremos para
# ver el espectro del archivo de música que acabamos de subir.
import numpy as np
import matplotlib.pyplot as plt
def plot_frec(x,fs,fmax,m): #esta función permite graficar las frecuencias de una señal x muestreada a tasa fs, el rango que grafica va hasta fmax, y m la  cantidad de puntos que toma para calcular la FFT .
  puntos = int(fmax/fs*m)
  lx = len(x)
  nt = (lx + m - 1)//m
  xp = np.append(x,np.zeros(-lx+nt*m))
  xmw = np.reshape( xp, (m,nt), order='F')
  xmf = np.fft.fft(xmw,len(xmw),axis=0)/m
  xf = np.linspace(0, int(fs/2.0), int(m/2))
  plt.plot(xf[0:puntos],np.abs(xmf[0:int(m/2),0:int(lx/m-1)]).mean(1)[0:puntos])
  plt.yscale('log')
  plt.show()

print("frecuencia de muestreo: ",fs)
plot_frec(data,fs,10000,4096)

In [ ]:
import numpy as np
#Repetimos aquí el código de los filtros que utilizaremos a continuación
import wave
import scipy.signal as signal
def filtro_pasabajo(x,f_corte,fs):
	taps = signal.firwin(151,f_corte,nyq=fs/2.0)
	filtered_x = signal.lfilter(taps, 1.0, x)
	return(filtered_x)

def filtro_pasaalto(x,f_corte,fs):
	taps = signal.firwin(151,f_corte, pass_zero=False,nyq=fs/2.0)
	filtered_x = signal.lfilter(taps, 1.0, x)
	return(filtered_x)

def filtro_pasabanda(x,f_corte1,f_corte2,fs):
	taps = signal.firwin(151, [f_corte1, f_corte2], pass_zero=False,nyq=fs/2.0)
	filtered_x = signal.lfilter(taps, 1.0, x)
	return(filtered_x)

In [ ]:
#Las radios AM para asegurarse que lo que transmiten no tenga un ancho mayor a 10 kHz y por lo tanto que no se superponga con las emisoras que tiene
# al lado, filtran previo a transmitir el audio con un filtro pasa bajo de 10 kHz.
#Nosotros vamos a filtrar el audio a 3 kHz, o sea vamos a pasar el audio por un filtro pasabajo de frecuencia de corte 3000 Hz.
data_filt = filtro_pasabajo(data,3000,fs)
# Graficaremos la señal filtrada y la escucharemos.
plot_frec(data_filt,fs,12000,4096)
Audio(data_filt,rate=fs)


Explique qué diferencias nota en el espectro de frecuencia de la señal filtrada y también de lo que se escucha respecto de la señal original.

Ahora vamos en modular en AM este audio, es decir lo vamos a mover para llevarlo a una frecuencia portadora $f_c$. Para eso comenzaremos haciendo una operación que es multiplicar la señal de audio filtrada por un coseno centrado en la frecuencia portadora $f_c$ que para nuestro caso la tomaremos arbitrariamente en 6 kHz.



In [ ]:
# En primer lugar generaremos un coseno de 10 KHz muestreado a una frecuencia de muestreo fs igual a la del audio filtrado que es de 24 kHz.
# Cuando vamos a multiplicar dos señales muesreadas en este caso el audio y el coseno ambos deben estar a la misma tasa de muestreo y
# tener la misma cantidad de muestras
f0 = 6000.0 # es la frecuencia portadora de nuestra radio AM
t=np.array([i for i in range(len(data_filt))])# genero un vector de tiempo
coseno = np.cos( 2*np.pi*f0/fs *t)
data_am = data_filt*coseno
plot_frec(data_am,fs,15000,4096)

Lo que sucede es que el espectro de la señal al multiplicarla por un coseno de frecuencia $f_c$ se traslada a la frecuencia $f_c$. En la figura observamos que el espectro que en el audio original estaba de 0 a 3 kHz ahora está de $f_c$ a $f_c$+3kHz.  Esto se llama modular en AM a la frecuencia $f_c$. Habrá observado que al modular la señal aparece un espectro aproximadamente simétrico en torno a la frecuencia de modulación $f_c$. Es decir que además del espectro que teníamos ahora aparece otro espejado respecto de $f_c$. Si el espectro del archivo wav lo observaramos incluyendo las “frecuencias negativas” también lo veríamos simétrico respecto a la frecuencia cero.

Puede resultar raro este concepto de frecuencias negativas pero no lo es tanto. La frecuencia $f_c$ la podemos definir por un vector que gira en el plano y da  $f_c$ vueltas por segundo en sentido antihorario. La frecuencia negativa la podemos imaginar como un vector que gira de la misma forma pero en sentido horario. Todas las señales reales tienen espectro positivo y negativo y además su espectro es simétrico respecto de la frecuencia cero. Pero eso ya lo verán más adelante en la carrera con detalle. Por ahora lo importante es entender que un coseno de frecuencia $f_c$ al multiplicarlo por cualquier señal traslada el espectro de la señal a esa frecuencia $f_c$.  

# Ejercicio 1 teórico para entregar
(Es un ejercicio teórico y además puede dar lugar a múltiples preguntas durante la realización de la entrevista final)

1. Busque información en internet y explique por qué a este proceso de multiplicar por un coseno de frecuencia más alta que el ancho de banda de la señal se denomina modulación en amplitud. Para visualizar este proceso, le recomendamos visualizarlo en el eje del tiempo realizando un ejemplo en Python que module un coseno de baja frecuencia por uno de frecuencia mucho mayor. Por ejemplo, tome un coseno de 100 Hz y multiplíquelo por un coseno de frecuencia 5 kHz ambas muestreadas a 40 kHz. Observe cómo se visualizan dos o tres períodos de la la señal modulada en el tiempo.


2. Realice un diagrama de bloques de cómo funcionaría el transmisor de Radio Nacional de España RNE (centrada en 657 kHz en Madrid y con un ancho de 10 kHz). Debajo se incluyen bloques típicos de sistemas de comunicaciones que le pueden ser de utilidad.

![diagrama_AM.png](https://iie.fing.edu.uy/~gbelcredi/tallerine/diagrama_AM.png)

![bloques.png](https://iie.fing.edu.uy/~gbelcredi/tallerine/bloques.png)


# **PARTE II**

# Algunas consideraciones sobre la frecuencia de muestreo: Teorema de muestreo
Ahora construiremos una señal que sea la suma de tres sinusoides a 1 kHz, 2 kHz, 3 kHz y con una tasa de muestreo de 20 kHz. Multiplicaremos esta señal por un coseno de la misma tasa de muestreo y de frecuencia portadora 6000 Hz.
Observe el espectro de la señal modulada y vea si obtiene el resultado esperado. Explique lo que observa.




In [ ]:
import numpy as np
fc = 6000.0 # es la frecuencia portadora de nuestra radio AM
fs = 20000
t=np.array([i for i in range(fs)])# genero un vector de tiempo
signal = np.sin( 2*np.pi*1000/fs *t)+ np.sin( 2*np.pi*2000/fs *t)+np.sin( 2*np.pi*3000/fs *t)

coseno = np.cos( 2*np.pi*fc/fs *t)
data_am = signal*coseno
plot_frec(data_am,fs,12000,4096)

Ahora modifique el código de la señal anterior y en lugar de ser una suma de senos de 1000, 2000 y 3000 Hz, haga una que sea la suma de 1000, 2000 y 5000 Hz y una última que sea la suma de 1000, 2000 y 6500 Hz y vea que pasa con los espectro de estas señales moduladas con el mismo coseno que antes. ¿Observa lo que esperaba?

In [ ]:
# celda para hacer la suma de los cosenos en 1000, 2000 y 5000 Hz modularlos en AM


In [ ]:
# celda para hacer la suma de los cosenos en 1000, 2000 y 6500 Hz modularlos en AM

Antes de tratar de explicar por qué sucede esto, discutamos qué es la tasa de muestreo que hemos utilizado. Un modelo de lo que sucede a lo largo del tiempo con, por ejemplo, el voltaje en un cable es pensar que tiene un valor para cualquier instante de tiempo t. Sea cual sea ese tiempo t, soy teóricamente capaz de medir el voltaje en ese instante y vale v(t) (o sea, una función en el tiempo t). Esto generalmente se denomina una señal analógica, un término que quizás haya escuchado nombrar. Seguramente también haya escuchado nombrar el término digital. En este contexto, una señal digital tiene dos diferencias fundamentales con respecto a v(t). La primera, y es la que exploramos ahora, es que no existe para cualquier t, sino para un conjunto de valores, tı́picamente (como en este caso) tomados cada tiempo fijo T (ver dibujo). Es decir, y por ejemplo, del audio en la computadora contamos únicamente con un conjunto de números v(0), v(T ), v(2T ), . . . , v(kT), ... Dado que la PC es un sistema digital, trabajamos con v(t) indirectamente a partir de este conjunto de muestras. La tasa de muestreo es la frecuencia a la que se toman estas muestras, es decir 1/T . Ver figura.

![figura muestreo](https://iie.fing.edu.uy/~gbelcredi/tallerine/muestreo.png)

Una pregunta fundamental de los sistemas de comunicaciones digitales es a qué frecuencia mínima puedo muestrear para poder reconstruir luego la señal original a partir de sus muestras. Parece intuitivo que si tomo muy pocas muestras de la señal, estoy perdiendo mucha información que luego no podré recuperar. Pero ¿cuál es esa frecuencia mínima de muestreo para no perder información de la señal original?

La respuesta la da el **Teorema de muestreo** que dice informalmente lo siguiente:
Si una señal analógica tiene en su espectro de frecuencias una frecuencia máxima $f_0$ (es decir que a partir de allí el espectro de frecuencia de la señal vale 0), entonces la frecuencia mínima de muestreo para poder reconstruir la señal a partir de sus muestras es $2 f_0$.

En el ejemplo con que iniciamos esta sección, cuando tenía tres sinusoides de 1 kHz, 3 kHz y 5 kHz y esa señal estaba modulada a 6 kHz, se tiene una señal cuya frecuencia máxima es 6+5 = 11 kHz y la señal estaba muestreada a 20 kHz, por lo tanto no se cumple el teorema de muestreo ya que 2*11 = 22 kHz sería la frecuencia mínima de muestreo y nosotros estabamos usando 20 kHz. Por eso le aparecían en la señal otras tonos en frecuencias que no deberían aparecer. Ese fenómeno se denomina aliasing o solapamiento.

Trataremos de dar una explicación intuitiva de lo que pasa cuando uno muestrea a una frecuencia menor a la que dice el teorema de muestreo.
Observemos la  siguiente figura. Obviamente la sinusoide original esta muestreada a una tasa menor a la del teorema de muestreo. Si yo solo tengo las muestras y quiero encontrar una sinusoide que pase por esos puntos, obviamente la sinusoide original pasa por esos puntos, pero también la dibujada en azul que es una sinusoide de frecuencia mucho menor. Probablemente pueda encontrar también sinusoides de frecuencia mucho mayor a la original que pasen por esos puntos. Es decir hay ambiguedad en la reconstrucción de la señal original. Sin embargo, si tengo varias muestras en un período de la señal eso intuitivamente se ve que no pasa.

![alt text](https://iie.fing.edu.uy/~belza/tallerinefiguras/aliasing.png)


Lo anterior se puede generalizar y extender a señales con un espectro contenido en el intervalo (−f, f ), como las que tiene el archivo wav analizado. Recuerde que dijimos que si bien nosotros visualizamos el eje positivo de frecuencias su espectro es simétrico en las frecuencias negativas. En ese caso se dice que la señal tiene ancho de banda f.
El denominado Teorema de Muestreo dice que, siempre que v(t) tenga ancho de banda f , es equivalente tener todo v(t) o sus muestras v(kT) siempre que 1/T > 2f, tal como ya dijimos.

Una observación importante es la siguiente. Las señales moduladas(AM,FM,TV,celular,etc) por ejemplo la FM a 94.7 MHz no se muestrean a esa frecuencia sino que primero se demodulan y se muestrea la señal demodulada en banda base es decir centrada en 0 Hz y no en 94.7. Por lo tanto el ancho de banda que se aplica para el muestro es el que tiene la señal original, voz, música, imagen o lo que fuera antes de ser llevada a alta frecuencia.

# Ejercicio 2 teórico para entregar
(Es un ejercicio teórico y además puede dar lugar a múltiples preguntas durante la realización de la entrevista final)

Busquen información sobre qué valores se pueden asignar en Sample Rate a la salida de audio de su PC. ¿Hasta que frecuencia se puede generar audio con estas tasas de muestreo? Tomando en cuenta el experimento que hizo en el Taller 3 referido al rango audible, ¿qué frecuencias de muestreo le parecen útiles?